# 3 · Bronze, from a stream

The last pipeline read **a table**. This one reads **a stream**, and almost
everything that is different about it comes from one fact:

> A table is a statement about how things **are**.
> A stream is a record of how they **got that way**.

| | |
|---|---|
| **reads** | Kafka topic `kerb.trips.lifecycle` |
| **writes** | `teach.bronze_events` in PostgreSQL |
| **runs** | every few minutes |

![](img/kafka-1-shape.png)

If the words **broker, topic, partition, offset, consumer group** are new, do
**notebook 10** first. It builds all of them from nothing. This notebook uses
them to write a pipeline.

In [ ]:
import sys; sys.path.insert(0, '.')
from nb import show, sql, fetch, run, counts

import json
import psycopg
from confluent_kafka import Consumer, TopicPartition
from pipelines.lib.config import dsn, SCHEMA, KAFKA, TOPIC_RIDES

print('broker :', KAFKA)
print('topic  :', TOPIC_RIDES)

---

## Step 0 · Look at the source before you touch it

Same rule as last time. For a stream that means: how many messages are there,
and how far behind are we.

In [ ]:
GROUP = 'teach-bronze-events'          # the name our position is stored under

def topic_facts():
    c = Consumer({'bootstrap.servers': KAFKA, 'group.id': GROUP,
                  'enable.auto.commit': False})
    parts = sorted(c.list_topics(TOPIC_RIDES, timeout=10).topics[TOPIC_RIDES].partitions)
    tps = [TopicPartition(TOPIC_RIDES, p) for p in parts]

    total = behind = 0
    print(f'{"partition":>10} {"messages":>12} {"our position":>14} {"behind":>10}')
    for tp, committed in zip(tps, c.committed(tps, timeout=10)):
        low, high = c.get_watermark_offsets(tp, timeout=10)
        at = committed.offset if committed.offset and committed.offset >= 0 else low
        total  += high - low
        behind += high - at
        print(f'{tp.partition:>10} {high - low:>12,} {at:>14,} {high - at:>10,}')
    c.close()
    print(f'{"total":>10} {total:>12,} {"":>14} {behind:>10,}')
    return behind

behind = topic_facts()

That last number has a name: **lag**. It is the single most watched number on
any streaming system, and it means *how many messages have been sent that we
have not read yet*.

## Read one message, and look at it

In [ ]:
c = Consumer({'bootstrap.servers': KAFKA, 'group.id': 'notebook-peek',
              'enable.auto.commit': False})

# go to the last message on partition 0: high watermark minus one
low, high = c.get_watermark_offsets(TopicPartition(TOPIC_RIDES, 0), timeout=10)
c.assign([TopicPartition(TOPIC_RIDES, 0, high - 1)])

msg = None
for _ in range(20):
    m = c.poll(1.0)
    if m is not None and not m.error():
        msg = m
        break
c.close()

print(f'partition {msg.partition()}, offset {msg.offset()}\n')
print(json.dumps(json.loads(msg.value()), indent=2))

### Note what a message is

**Bytes.** Not a row, not a dict, not a typed object. Somebody else's code
produced that JSON and pushed it, and nothing checked it on the way. It might
not be JSON at all. It might be JSON with the fields missing.

Everything below is written knowing that.

---

## Step 1 · The table, with a key that makes replays harmless

Look at the primary key. It is not one id column, it is **the pair
`(trip_id, event)`**.

In [ ]:
DDL = f"""
CREATE TABLE IF NOT EXISTS {SCHEMA}.bronze_events (
    trip_id     TEXT NOT NULL,
    event       TEXT NOT NULL,        -- requested, accepted, driver_arrived, started, completed
    happened_at TIMESTAMPTZ,
    driver_id   TEXT,
    fare        NUMERIC(10,2),        -- only present on the 'completed' event
    PRIMARY KEY (trip_id, event)      -- the whole reason a replay is boring
);
"""

with psycopg.connect(dsn(), autocommit=True) as c:
    c.execute(DDL)

print('table ready')

That key is **a statement about the real world**: a given ride can only ever
have one `completed` event.

Saying it in the table definition means the database itself refuses a duplicate.
We do not write deduplication code, and we cannot forget to.

---

## Step 2 · The contract for a message

In [ ]:
REQUIRED = ('trip_id', 'event', 'ts')

def parse(raw_bytes):
    """Turn one raw message into a row, or explain why it cannot be one.

    Returns (row, None) or (None, reason). Never raises: one poison message
    must not stop the four hundred thousand behind it.
    """
    try:
        d = json.loads(raw_bytes)
        missing = [k for k in REQUIRED if d.get(k) in (None, '')]
        if missing:
            return None, f"missing required field(s): {', '.join(missing)}"
    except Exception as e:
        return None, str(e)

    # .get() for the optional fields, [] for the required ones. That is not
    # style: a required field missing here would already have been caught above.
    return (d['trip_id'], d['event'], d['ts'], d.get('driver_id'), d.get('fare')), None

print('a real message :', parse(msg.value()))
print()
print('no ts          :', parse(b'{"trip_id":"TRP1","event":"started","ts":""}'))
print('not even json  :', parse(b'<html>502 Bad Gateway</html>'))

**Three inputs, three different answers, no exception raised.** The good one
becomes a row. The other two come back with a reason a human can read.

---

## Give the pipeline something to read

The pipeline may already be caught up, in which case the loop below reads
nothing and proves nothing. So put ten rides on the topic first.

In [ ]:
from confluent_kafka import Producer
import datetime as dt, random

producer = Producer({'bootstrap.servers': KAFKA})
LIFECYCLE = ['requested', 'accepted', 'driver_arrived', 'started', 'completed']
now = dt.datetime.now(dt.timezone.utc).replace(microsecond=0)

sent = 0
for _ in range(10):
    trip_id = f'TRP-NB3-{random.randint(10000, 99999)}'
    driver  = f'DRV{random.randint(1, 2800):06d}'
    for i, name in enumerate(LIFECYCLE):
        e = {'trip_id': trip_id, 'event': name,
             'ts': (now + dt.timedelta(minutes=i * 3)).isoformat(),
             'driver_id': None if name == 'requested' else driver,
             'pu_zone_id': random.randint(1, 60)}
        if name == 'completed':
            e['fare'] = round(random.uniform(60, 420), 2)
        # same key every time, so all five events of a ride stay on one partition
        producer.produce(TOPIC_RIDES, key=trip_id.encode(), value=json.dumps(e).encode())
        sent += 1

# one message nobody can read, so the held pile has something to do
producer.produce(TOPIC_RIDES, key=b'TRP-NB3-BROKEN',
                 value=json.dumps({'trip_id': 'TRP-NB3-BROKEN',
                                   'event': 'started', 'ts': ''}).encode())
producer.flush(10)

print(f'{sent} good events and 1 unreadable one are now on the topic')

In [ ]:
behind = topic_facts()

---

## Step 3 · Open a consumer, with auto-commit turned OFF

Four lines of configuration, and one of them is the most important line in the
whole pipeline.

In [ ]:
consumer = Consumer({
    'bootstrap.servers': KAFKA,
    'group.id': GROUP,                # the name our position is stored under
    'auto.offset.reset': 'earliest',  # never read before? start at the beginning
    'enable.auto.commit': False,      # WE commit, after the write. See below.
})
consumer.subscribe([TOPIC_RIDES])

print('subscribed as', GROUP)

### Why `enable.auto.commit = False` is the line that matters

By default the Kafka library quietly saves your position **on a timer, in the
background**, whether or not your database write succeeded. Picture the order:

1. library reads 5,000 messages
2. the library's timer fires and saves *"I have read up to here"*
3. your process dies before writing them to Postgres

Those 5,000 messages are **gone**. Not delayed. Gone. The next run starts after
them, and nobody will ever know they existed, because there is no error and no
gap that anything can detect.

![](img/kafka-4-commit.png)

---

## Step 4 · Read, write, and only then save your position

Here is the whole loop. Read it, then run it.

In [ ]:
INSERT = f"""
    INSERT INTO {SCHEMA}.bronze_events (trip_id, event, happened_at, driver_id, fare)
    VALUES (%s, %s, %s, %s, %s)
    ON CONFLICT DO NOTHING          -- the same (trip, event) twice is fine, keep the first
"""

read_n = written = held = 0

while True:
    # Ask for a batch. Nothing within 3 seconds means we have caught up, and
    # catching up is how this pipeline ends.
    batch = consumer.consume(num_messages=5000, timeout=3.0)
    if not batch:
        break

    rows = []
    for m in batch:
        if m.error():
            continue
        read_n += 1
        row, reason = parse(m.value())
        if row is None:
            held += 1                      # the real pipeline writes these to quarantine
        else:
            rows.append(row)

    if rows:
        # FIRST: make the rows durable, in their own transaction.
        with psycopg.connect(dsn(), autocommit=False) as c, c.cursor() as cur:
            cur.executemany(INSERT, rows)
            c.commit()
        written += len(rows)

    # ONLY NOW is it true that we have consumed these messages, so only now do
    # we say so. asynchronous=False waits for the broker to confirm.
    consumer.commit(asynchronous=False)
    print(f'  batch: read {len(batch):,}  wrote {len(rows):,}  committed')

consumer.close()
print(f'\ncaught up. read {read_n:,}, wrote {written:,}, held {held:,}')

## Are we caught up?

In [ ]:
behind = topic_facts()

---

## Now prove a replay is harmless

This is the property the primary key bought us. Insert **the exact same rows
again** and watch the table not change.

In [ ]:
before = fetch(f'SELECT count(*) AS n FROM {SCHEMA}.bronze_events').n[0]

# take 1,000 rows straight back out of the table and insert them again
with psycopg.connect(dsn()) as c:
    same = c.execute(f"""SELECT trip_id, event, happened_at, driver_id, fare
                        FROM {SCHEMA}.bronze_events LIMIT 1000""").fetchall()

with psycopg.connect(dsn(), autocommit=False) as c, c.cursor() as cur:
    cur.executemany(INSERT, same)
    c.commit()

after = fetch(f'SELECT count(*) AS n FROM {SCHEMA}.bronze_events').n[0]
print(f'before  {before:,}')
print(f'after   {after:,}   (we just re-inserted 1,000 rows)')
print('\nno change' if before == after else 'DIFFERENT, the key is not doing its job')

### That is why we can afford at-least-once

> **Duplicates you can remove. Missing data you cannot invent.**

Reading a message more than once is called **at-least-once** delivery, and it is
what almost every real streaming pipeline chooses. It only works because the
table refuses the second copy.

---

## And now the packaged pipeline

Same loop, plus quarantine and the run log.

In [ ]:
run('-m', 'pipelines.p2_bronze_events')

In [ ]:
sql(f"""
    SELECT pipeline, status, rows_in, rows_out,
           round(extract(epoch from (ended_at - started_at))::numeric, 2) AS secs, message
    FROM {SCHEMA}.runs
    WHERE pipeline = 'p2_bronze_events'
    ORDER BY started_at DESC LIMIT 3
""", 'the run log')

## What one ride looks like once it has landed

In [ ]:
sql(f"""
    SELECT trip_id, event, happened_at, driver_id, fare
    FROM {SCHEMA}.bronze_events
    WHERE trip_id = (SELECT trip_id FROM {SCHEMA}.bronze_events
                     WHERE event = 'completed' ORDER BY happened_at DESC LIMIT 1)
    ORDER BY happened_at
""", 'every event of one ride, in order')

**Five rows, one ride.** That is the difference from notebook 2, in one table.
`bronze_trips` has one row saying how that ride ended. `bronze_events` has the
whole story of how it got there.

---

## What you learned

- A stream is **bytes somebody else produced**. Nothing checked it on the way
- **Lag** is how many messages have been sent that you have not read
- The primary key `(trip_id, event)` is **a statement about the real world**,
  and it is what makes a replay boring
- `enable.auto.commit = False`, always. **You** decide when the position moves
- **Write the rows, then commit the offset.** Never the other way round
- `ON CONFLICT DO NOTHING` plus that key is the entire deduplication strategy
- One poison message must never block the ones behind it